# Language Model Foundations

Build a tiny character-level next-character model from a local string. Inspect token IDs, shifted targets, logits, and cross-entropy without downloading text or weights.

In [ ]:
import torch
from torch import nn

torch.manual_seed(41)
torch.set_num_threads(1)
print('CPU | character next-token example')

In [ ]:
text = 'deep learning ' * 20
alphabet = sorted(set(text))
to_id = {character: index for index, character in enumerate(alphabet)}
tokens = torch.tensor([to_id[character] for character in text])
context = 8
inputs = torch.stack([tokens[index:index+context] for index in range(len(tokens)-context)])
targets = torch.stack([tokens[index+1:index+context+1] for index in range(len(tokens)-context)])
assert inputs.shape == targets.shape and inputs.shape[1] == context
print('vocabulary:', ''.join(alphabet), '| examples:', len(inputs), '| context:', context)

In [ ]:
model = nn.Sequential(nn.Embedding(len(alphabet), 16), nn.Flatten(),
                      nn.Linear(context * 16, len(alphabet)))
optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
for _ in range(100):
    selection = torch.randint(len(inputs), (32,))
    logits = model(inputs[selection])
    loss = nn.functional.cross_entropy(logits, targets[selection, -1])
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
assert torch.isfinite(loss)
print(f'next-character loss={loss.item():.4f}')
print('predicted next character:', alphabet[model(inputs[:1]).argmax(1).item()])

The repeated training string lets this small model memorize its patterns, which is why its training loss can approach zero. There is no held-out text here; this result does not measure generalization or language understanding.